# 1. Instalaciones

In [ ]:
%pip install azure-ai-documentintelligence azure-storage-blob python-dotenv azure-identity
%pip install pandas numpy tqdm  openpyxl xlrd tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Importaciones

In [ ]:
import os
import json
import time
import re
from datetime import datetime
from typing import Dict, Any, List
import traceback

# Azure
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.storage.blob import BlobServiceClient, ContentSettings

# Utilidades
from dotenv import load_dotenv
import pandas as pd

# 3. Configuración

In [2]:
load_dotenv()

CONNECTION_STRING = os.getenv("BLOB_CONNECTION_STRING")
CONTAINER_NAME = os.getenv("BRONZE_CONTAINER_NAME")
SOURCE_PREFIX = os.getenv("BLOB_SOURCE_PREFIX", "servicio_policia/")
EXTRACTED_PREFIX = os.getenv("BLOB_EXTRACTED", "servicio_policia/servicio_policia_extracted/")

# Credenciales de Document Intelligence en Azure
DOCUMENT_INTELLIGENCE_ENDPOINT = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
DOCUMENT_INTELLIGENCE_KEY = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY")


# Validación de variables críticas
required_vars = ["BLOB_CONNECTION_STRING", "BRONZE_CONTAINER_NAME", "AZURE_DOCUMENT_INTELLIGENCE_KEY", "AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Faltan variables de entorno: {', '.join(missing)}")

# Inicializar cliente Storage
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# Inicializar cliente Document Intelligence
doc_intel_client = DocumentIntelligenceClient(
    endpoint=DOCUMENT_INTELLIGENCE_ENDPOINT,
    credential=AzureKeyCredential(DOCUMENT_INTELLIGENCE_KEY)
)

In [3]:
print(f'Fuente: {SOURCE_PREFIX}')

print(f'Destino: {EXTRACTED_PREFIX}')

Fuente: servicio_policia/
Destino: servicio_policia/servicio_policia_extracted/


# 4. Funciones de extracción

In [5]:
def extract_basic_metadata(file_path: str, blob_properties: Any) -> Dict[str, Any]:
    """Extrae metadatos básicos del archivo"""
    file_extension = os.path.splitext(file_path)[1].lower()
    
    metadata = {
        "file_name": os.path.basename(file_path),
        "file_path": file_path,
        "file_extension": file_extension
    }
    
    return metadata

# 5. Función para  procesar archivos excel

In [6]:
def process_excel_file(file_content: bytes, file_extension: str) -> Dict[str, Any]:
    """Procesa archivos Excel (XLSX, XLS) usando pandas"""
    try:
        if file_extension == '.xlsx':
            excel_file = pd.ExcelFile(file_content, engine='openpyxl')
        elif file_extension == '.xls':
            excel_file = pd.ExcelFile(file_content, engine='xlrd')
        else:
            return {
                "status": "error",
                "error_message": f"Formato Excel no soportado: {file_extension}"
            }
        
        # Extraer contenido de todas las hojas
        sheets_data = {}
        plain_text = ""
        markdown_content = "# Contenido del archivo Excel\n\n"
        
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            
            # Convertir a texto plano
            rows_as_text = "\n".join(
                df.fillna("")
                .astype(str)
                .apply(lambda r: " | ".join(r), axis=1)
            )

            sheet_text = f"Hoja: {sheet_name}\n{rows_as_text}\n\n"
            plain_text += sheet_text
            
            # Convertir a markdown
            sheet_markdown = f"## Hoja: {sheet_name}\n\n{df.to_markdown(index=False)}\n\n"
            markdown_content += sheet_markdown
            
            # Guardar datos de la hoja
            sheets_data[sheet_name] = {
                "shape": df.shape,
                "columns": df.columns.tolist(),
                "head": df.head().to_dict('records')
            }
        
        return {
            "status": "success",
            "processed_result": {
                "content": {
                    "plain_text": plain_text,
                    "markdown": markdown_content
                },
                "sheets": sheets_data,
                "total_sheets": len(excel_file.sheet_names),
                "document_summary": {
                    "total_pages": len(excel_file.sheet_names),
                    "total_tables": len(excel_file.sheet_names),
                    "total_key_value_pairs": 0
                }
            },
            "processing_details": {
                "model": "pandas_excel_processor",
                "timestamp": datetime.utcnow().isoformat()
            }
        }
        
    except Exception as e:
        error_msg = f"Error procesando archivo Excel: {str(e)}"
        print(error_msg)
        traceback.print_exc()
        return {
            "status": "error",
            "error_message": str(e),
            "processing_details": {
                "model": "pandas_excel_processor",
                "timestamp": datetime.utcnow().isoformat()
            }
        }

# 6. Funciones de Document Intelligence

In [ ]:
def analyze_document_with_doc_intel(file_content: bytes, file_extension: str) -> Dict[str, Any]:
    """Analiza documento usando Azure Document Intelligence"""
    
    try:
        # Preparar el contenido según el tipo de archivo
        if file_extension == '.pdf':
            content_type = "application/pdf"
        elif file_extension == '.docx':
            content_type = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
        elif file_extension == '.doc':
            content_type = "application/msword"
        else:
            return {
                "status": "error",
                "error_message": f"Formato no soportado por Document Intelligence: {file_extension}"
            }
        
        # Llamar a Document Intelligence
        poller = doc_intel_client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=file_content,
            content_type=content_type
        )
        
        # Esperar resultados
        result = poller.result()
        
        # Procesar resultados con offset_map
        processed_result = process_document_intelligence_result_with_offset_map(result)
        
        return {
            "status": "success",
            "processed_result": processed_result,
            "processing_details": {
                "model": "prebuilt-layout",
                "timestamp": datetime.utcnow().isoformat()
            }
        }
        
    except Exception as e:
        error_msg = f"Error en Document Intelligence: {str(e)}"
        print(error_msg)
        traceback.print_exc()
        return {
            "status": "error",
            "error_message": str(e),
            "processing_details": {
                "model": "prebuilt-layout",
                "timestamp": datetime.utcnow().isoformat()
            }
        }

def process_document_intelligence_result_with_offset_map(result) -> Dict[str, Any]:
    """Procesa y estructura los resultados de Document Intelligence con offset_map"""
    
    try:
        if result is None:
            return {"error": "El resultado de Document Intelligence es None"}
        
        processed_data = {
            "content": {
                "markdown": "",
                "plain_text": "",
            },
            "pages": [],
            "tables": [],
            "key_value_pairs": [],
            "languages": [],
            "styles": [],
            "document_summary": {},
            "offset_map": []
        }
        
        # 1. Extraer contenido markdown (si está disponible)
        if hasattr(result, 'content') and result.content:
            processed_data["content"]["markdown"] = result.content
        elif hasattr(result, 'markdown_content') and result.markdown_content:
            processed_data["content"]["markdown"] = result.markdown_content
        
        # 2. Extraer texto plano
        plain_text = extract_plain_text_from_result(result)
        processed_data["content"]["plain_text"] = plain_text
        
        # 3. Generar offset_map
        offset_map = generate_offset_map(result)
        processed_data["offset_map"] = offset_map
        
        # 4. Extraer información de páginas
        if hasattr(result, 'pages') and result.pages:
            for i, page in enumerate(result.pages):
                lines = page.lines or []
                words = page.words or []

                page_data = {
                    "page_number": i + 1,
                    "angle": getattr(page, 'angle', 0),
                    "width": getattr(page, 'width', 0),
                    "height": getattr(page, 'height', 0),
                    "unit": getattr(page, 'unit', 'pixel'),
                    "lines_count": len(lines),
                    "words_count": len(words),
                }
                processed_data["pages"].append(page_data)
        
        # 5. Extraer tablas
        if hasattr(result, 'tables') and result.tables:
            for i, table in enumerate(result.tables):
                table_data = {
                    "table_number": i + 1,
                    "row_count": getattr(table, 'row_count', 0),
                    "column_count": getattr(table, 'column_count', 0),
                }
                processed_data["tables"].append(table_data)
        
        # 6. Extraer pares clave-valor
        if hasattr(result, 'key_value_pairs') and result.key_value_pairs:
            for kv in result.key_value_pairs:
                if hasattr(kv, 'key') and hasattr(kv, 'value'):
                    kv_data = {
                        "key": getattr(kv.key, 'content', '') if kv.key else '',
                        "value": getattr(kv.value, 'content', '') if kv.value else '',
                        "confidence": getattr(kv, 'confidence', 0)
                    }
                    processed_data["key_value_pairs"].append(kv_data)
        
        # 7. Extraer idiomas detectados
        if hasattr(result, 'languages') and result.languages:
            for lang in result.languages:
                lang_data = {
                    "locale": getattr(lang, 'locale', ''),
                    "confidence": getattr(lang, 'confidence', 0),
                }
                processed_data["languages"].append(lang_data)
        
        # 8. Extraer estilos de fuente
        if hasattr(result, 'styles') and result.styles:
            for style in result.styles:
                style_data = {
                    "is_handwritten": getattr(style, 'is_handwritten', False),
                    "confidence": getattr(style, 'confidence', 0),
                }
                processed_data["styles"].append(style_data)
        
        # 9. Crear resumen del documento
        processed_data["document_summary"] = {
            "total_pages": len(processed_data["pages"]),
            "total_tables": len(processed_data["tables"]),
            "total_key_value_pairs": len(processed_data["key_value_pairs"]),
            "content_length": len(processed_data["content"]["markdown"]),
            "has_handwritten_text": any(style.get("is_handwritten", False) for style in processed_data["styles"])
        }
        
        return processed_data
        
    except Exception as e:
        print(f"Error procesando resultados de Document Intelligence: {str(e)}")
        traceback.print_exc()
        return {"error": str(e)}

def generate_offset_map(result) -> List[Dict[str, Any]]:
    """Genera el offset_map basado en la lógica de document_processor.py"""
    
    offset_map = []
    
    if hasattr(result, 'paragraphs') and result.paragraphs:
        current_section = None
        
        for p in result.paragraphs:
            page_num = 1
            if hasattr(p, 'bounding_regions') and p.bounding_regions:
                page_num = p.bounding_regions[0].page_number
            
            # Detección de secciones
            is_header = False
            
            # Check 1: Usando role
            if hasattr(p, 'role') and p.role in ['sectionHeading', 'title']:
                is_header = True
                current_section = p.content if hasattr(p, 'content') else None
            
            # Check 2: Heurísticas para DOCX/Layouts sin roles
            if not is_header and hasattr(p, 'content') and p.content:
                content_clean = p.content.strip()
                if content_clean:
                    # A. Headers numerados
                    is_numbered = bool(re.match(r'^(\d+(\.\d+)*|[IVX]+(\.[IVX]+)*)\.?\s+[A-ZÁÉÍÓÚÑ]', content_clean))
                    
                    # B. Títulos en mayúsculas
                    is_caps = (len(content_clean) < 150 and len(content_clean) > 3 
                            and content_clean.isupper() 
                            and not content_clean.replace('.','').replace(' ','').isdigit())

                    if is_numbered or is_caps:
                        is_header = True
                        current_section = content_clean

            # Sanitizar current_section si fue establecido/actualizado
            if is_header and current_section:
                current_section = current_section.replace('\n', ' ').strip()
                if len(current_section) > 100: 
                    current_section = current_section[:100] + "..."

            # Agregar offsets para cada span
            if hasattr(p, 'spans') and p.spans:
                for span in p.spans:
                    offset_map.append({
                        "start": span.offset,
                        "end": span.offset + span.length,
                        "page": page_num,
                        "section": current_section
                    })
    
    # Ordenar por start offset
    offset_map.sort(key=lambda x: x["start"])
    
    return offset_map

def extract_plain_text_from_result(result) -> str:
    """Extrae texto plano de los resultados de Document Intelligence"""
    try:
        if result is None:
            return ""
        
        text_parts = []
        
        # Método 1: Intentar obtener el contenido directo
        if hasattr(result, 'content') and result.content:
            return result.content
        
        # Método 2: Extraer de páginas y líneas
        if hasattr(result, 'pages') and result.pages:
            for page in result.pages:
                if hasattr(page, 'lines') and page.lines:
                    for line in page.lines:
                        if hasattr(line, 'content') and line.content:
                            text_parts.append(line.content)
        
        return "\n".join(text_parts).strip()
    
    except Exception as e:
        print(f"Error extrayendo texto plano: {str(e)}")
        return ""

# 7. Procesamiento principal

In [ ]:
def process_document_with_doc_intel(blob_name: str) -> Dict[str, Any]:
    """Procesa un documento individual usando Azure Document Intelligence o pandas para Excel"""
    
    try:
        # Descargar archivo
        blob_client = container_client.get_blob_client(blob_name)
        blob_properties = blob_client.get_blob_properties()
        
        # Extraer metadatos básicos
        basic_metadata = extract_basic_metadata(blob_name, blob_properties)

        # Descargar contenido
        download_stream = blob_client.download_blob()
        file_content = download_stream.readall()
        
        # Procesar según tipo de archivo
        file_extension = os.path.splitext(blob_name)[1].lower()

        if file_extension in ['.pdf', '.docx', '.doc']:
            print(f"  Procesando con Document Intelligence: {blob_name}")
            
            # Procesar con Document Intelligence
            doc_intel_result = analyze_document_with_doc_intel(file_content, file_extension)
            
            # Preparar resultado con valores por defecto
            extraction_results = {
                "content": "",  # CAMBIO: Ahora solo "content" en lugar de "document_intelligence_text"
            }
            document_analysis = {
                "total_pages": 0,
                "total_tables": 0,
                "total_key_value_pairs": 0
            }
            
            # Solo procesar si fue exitoso
            if doc_intel_result.get('status') == 'success':
                processed = doc_intel_result.get('processed_result', {})
                if processed:
                    content = processed.get('content', {})
                    # Usar Markdown si existe y no está vacío, de lo contrario texto plano
                    markdown = content.get('markdown', '').strip()
                    plain = content.get('plain_text', '').strip()
                    selected_content = markdown if markdown else plain
                    extraction_results = {
                        "content": selected_content
                    }
                    document_summary = processed.get('document_summary', {})
                    document_analysis = {
                        "total_pages": document_summary.get('total_pages', 0),
                        "total_tables": document_summary.get('total_tables', 0),
                        "total_key_value_pairs": document_summary.get('total_key_value_pairs', 0)
                    }
            
            # Obtener offset_map
            offset_map = processed.get('offset_map', []) if doc_intel_result.get('status') == 'success' else []
            
            result = {
                "basic_metadata": basic_metadata,
                "extraction_results": extraction_results,  # CAMBIO: Ahora solo contiene "content"
                "document_analysis": document_analysis,
                "detailed_analysis": {
                    "pages_count": document_analysis.get('total_pages', 0),
                    "tables_count": document_analysis.get('total_tables', 0),
                    "key_value_pairs_count": document_analysis.get('total_key_value_pairs', 0)
                },
                "offset_map": offset_map
            }
            
            return result
            
        elif file_extension in ['.xlsx', '.xls']:
            print(f"  Procesando con pandas (Excel): {blob_name}")
            
            # Procesar con pandas para Excel
            excel_result = process_excel_file(file_content, file_extension)
            
            if excel_result.get('status') == 'success':
                processed = excel_result.get('processed_result', {})
                content = processed.get('content', {})
                document_summary = processed.get('document_summary', {})
                
                # Usar Markdown si existe y no está vacío, de lo contrario texto plano
                markdown = content.get('markdown', '').strip()
                plain = content.get('plain_text', '').strip()
                selected_content = markdown if markdown else plain
                
                result = {
                    "basic_metadata": basic_metadata,
                    "extraction_results": {
                        "content": selected_content,
                    },
                    "document_analysis": {
                        "total_pages": document_summary.get('total_pages', 0),
                        "total_tables": document_summary.get('total_tables', 0),
                        "total_key_value_pairs": document_summary.get('total_key_value_pairs', 0)
                    },
                    "detailed_analysis": {
                        "pages_count": document_summary.get('total_pages', 0),
                        "tables_count": document_summary.get('total_tables', 0),
                        "key_value_pairs_count": document_summary.get('total_key_value_pairs', 0)
                    },
                    "offset_map": []  # Excel no tiene offset_map
                }
                return result
            else:
                return {
                    "basic_metadata": basic_metadata,
                    "error": excel_result.get('error_message', 'Error desconocido'),
                    "extraction_results": {"content": ""},
                    "document_analysis": {
                        "total_pages": 0,
                        "total_tables": 0,
                        "total_key_value_pairs": 0
                    },
                    "detailed_analysis": {
                        "pages_count": 0,
                        "tables_count": 0,
                        "key_value_pairs_count": 0
                    },
                    "offset_map": []
                }
            
        else:
            error_msg = f"Formato no soportado: {file_extension}"
            print(f"  {error_msg}")
            
            return {
                "basic_metadata": basic_metadata,
                "error": error_msg,
                "extraction_results": {"content": ""},
                "document_analysis": {
                    "total_pages": 0,
                    "total_tables": 0,
                    "total_key_value_pairs": 0
                },
                "detailed_analysis": {
                    "pages_count": 0,
                    "tables_count": 0,
                    "key_value_pairs_count": 0
                },
                "offset_map": []
            }
    
    except Exception as e:
        print(f"  ✗ Error procesando {blob_name}: {str(e)}")
        traceback.print_exc()
        
        return {
            "basic_metadata": {"file_name": blob_name, "error": str(e)},
            "extraction_results": {"content": ""},
            "document_analysis": {
                "total_pages": 0,
                "total_tables": 0,
                "total_key_value_pairs": 0
            },
            "detailed_analysis": {
                "pages_count": 0,
                "tables_count": 0,
                "key_value_pairs_count": 0
            },
            "offset_map": []
        }

# 8. Guardar resultados

In [9]:
def save_extraction_result(original_blob_name: str, result: Dict[str, Any]):
    """Guarda el resultado en Blob Storage"""
    
    # Extraer solo el nombre del archivo
    filename = os.path.basename(original_blob_name)
    
    # Crear el nuevo nombre: nombre base + .json
    base_name = os.path.splitext(filename)[0]
    json_name = f"{base_name}.json"
    
    # Añadir el prefijo de destino
    json_blob_name = f"{EXTRACTED_PREFIX}{json_name}"
    
    # Convertir a JSON
    def default_serializer(obj):
        if hasattr(obj, 'isoformat'):
            return obj.isoformat()
        return str(obj)
    
    json_content = json.dumps(result, indent=2, ensure_ascii=False, default=default_serializer)
    
    # Subir a Blob Storage
    blob_client = container_client.get_blob_client(json_blob_name)
    blob_client.upload_blob(
        json_content,
        overwrite=True,
        content_settings=ContentSettings(content_type='application/json')
    )
    
    return json_blob_name

def process_documents_batch(blob_list=None, max_documents=500, delay_between_requests=2):
    """Procesa un lote de documentos con retraso entre solicitudes"""
    
    if blob_list is None:
        # Listar blobs en el prefijo de origen
        blobs = container_client.list_blobs(name_starts_with=SOURCE_PREFIX)
        # Incluir todos los formatos soportados
        supported_formats = ['.pdf', '.docx', '.xlsx', '.xls']

        #supported_formats =  ['2ij-pr-0034 investigar delitos.xlsx']            #BIEN -----> XLSX
        #supported_formats =  ['2ij-pr-0024 realizar entrevista.xls']            #BIEN -----> XLS
        #supported_formats = ['ley_1257_de_2008.pdf']                            #BIEN -----> PDF
        #supported_formats = ['información de apoyo cursos mandatorios.docx']    #BIEN -----> DOCX
        #supported_formats = ['fpj-01 reporte de iniciacion.docx']               #BIEN -----> DOCX (convertido)
        
        blob_list = [blob.name for blob in blobs if any(blob.name.lower().endswith(ext) for ext in supported_formats)]
    
    # Limitar el número de documentos si se especifica
    if max_documents:
        blob_list = blob_list[:max_documents]
    
    processed = []
    errors = []
    
    print(f"Iniciando procesamiento de {len(blob_list)} documentos...")
    print(f"Formatos soportados: PDF, DOCX, XLSX, XLS")
    print("-" * 60)
    
    for i, blob_name in enumerate(blob_list, 1):
        try:
            print(f"[{i}/{len(blob_list)}] Procesando: {blob_name}")
            
            # Procesar documento
            result = process_document_with_doc_intel(blob_name)
            
            # Verificar si hubo error
            if result.get('error') or (result.get('extraction_results', {}).get('content') == '' and result.get('document_analysis', {}).get('total_pages') == 0):
                errors.append({
                    "blob": blob_name,
                    "error": result.get('error', 'Error desconocido o contenido vacío'),
                    "success": False
                })
                print(f"   ✗ Error en procesamiento")
                continue
            
            # Guardar resultado
            json_name = save_extraction_result(blob_name, result)
            
            processed.append({
                "original": blob_name,
                "result": json_name,
                "success": True,
                "pages": result.get('document_analysis', {}).get('total_pages', 0)
            })
            
            print(f"   ✓ Guardado: {json_name}")
            print(f"   Páginas/hojas procesadas: {result.get('document_analysis', {}).get('total_pages', 0)}")
            print(f"   Longitud del contenido: {len(result.get('extraction_results', {}).get('content', ''))} caracteres")
            
            # Esperar entre solicitudes para evitar límites de tasa
            if i < len(blob_list):
                time.sleep(delay_between_requests)
            
        except Exception as e:
            error_info = {
                "blob": blob_name,
                "error": str(e),
                "success": False,
                "traceback": traceback.format_exc()
            }
            errors.append(error_info)
            print(f"   ✗ Error crítico: {str(e)}")
    
    return processed, errors

# 9. Ejecución

In [10]:
processed, errors = process_documents_batch(max_documents=500)

print("\n" + "="*60)
print("RESUMEN DE PROCESAMIENTO")
print("="*60)
print(f"Documentos procesados exitosamente: {len(processed)}")
print(f"Documentos con errores: {len(errors)}")

if processed:
    print("\nÚltimo documento procesado exitosamente:")
    print(f"  Archivo: {processed[-1]['original']}")
    print(f"  JSON: {processed[-1]['result']}")

if errors:
    print("\nÚltimo error encontrado:")
    print(f"  Archivo: {errors[-1]['blob']}")
    print(f"  Error: {errors[-1]['error'][:100]}...")

Iniciando procesamiento de 234 documentos...
Formatos soportados: PDF, DOCX, XLSX, XLS
------------------------------------------------------------
[1/234] Procesando: servicio_policia/11001-03-06-000-2010-00097-00(2034).pdf
  Procesando con Document Intelligence: servicio_policia/11001-03-06-000-2010-00097-00(2034).pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/11001-03-06-000-2010-00097-00(2034).json
   Páginas/hojas procesadas: 16
   Longitud del contenido: 51498 caracteres
[2/234] Procesando: servicio_policia/12. Decreto 2535 de 1993. Normas armas, municiones y explosivos.pdf
  Procesando con Document Intelligence: servicio_policia/12. Decreto 2535 de 1993. Normas armas, municiones y explosivos.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/12. Decreto 2535 de 1993. Normas armas, municiones y explosivos.json
   Páginas/hojas procesadas: 10
   Longitud del contenido: 19713 caracteres
[3/234] Procesando: servicio_policia/1CS-MA-0001 MANUAL PARA EL SERV

C:\Users\NicolayCastellanosPe\AppData\Local\Temp\ipykernel_41980\3074985718.py:5: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  excel_file = pd.ExcelFile(file_content, engine='openpyxl')


   ✓ Guardado: servicio_policia/servicio_policia_extracted/Estandar de capacidades y competencias del profesional de Policía.json
   Páginas/hojas procesadas: 2
   Longitud del contenido: 199087 caracteres
[11/234] Procesando: servicio_policia/Formatos Policía Judicial/FPJ-01 REPORTE DE INICIACION.docx
  Procesando con Document Intelligence: servicio_policia/Formatos Policía Judicial/FPJ-01 REPORTE DE INICIACION.docx
   ✓ Guardado: servicio_policia/servicio_policia_extracted/FPJ-01 REPORTE DE INICIACION.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 833 caracteres
[12/234] Procesando: servicio_policia/Formatos Policía Judicial/FPJ-02 NOTICIA CRIMINAL.docx
  Procesando con Document Intelligence: servicio_policia/Formatos Policía Judicial/FPJ-02 NOTICIA CRIMINAL.docx
   ✓ Guardado: servicio_policia/servicio_policia_extracted/FPJ-02 NOTICIA CRIMINAL.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 8724 caracteres
[13/234] Procesando: servicio_policia/Formatos

C:\Users\NicolayCastellanosPe\AppData\Local\Temp\ipykernel_41980\3074985718.py:5: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  excel_file = pd.ExcelFile(file_content, engine='openpyxl')


   ✓ Guardado: servicio_policia/servicio_policia_extracted/FPJ-28 Acta-de-Consentimiento.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 5959 caracteres
[35/234] Procesando: servicio_policia/Formatos Policía Judicial/FPJ-30 Acta de Entrega.pdf
  Procesando con Document Intelligence: servicio_policia/Formatos Policía Judicial/FPJ-30 Acta de Entrega.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/FPJ-30 Acta de Entrega.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 1374 caracteres
[36/234] Procesando: servicio_policia/Formatos Policía Judicial/FPJ-31 Derechos y Deberes de las Víctimas.pdf
  Procesando con Document Intelligence: servicio_policia/Formatos Policía Judicial/FPJ-31 Derechos y Deberes de las Víctimas.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/FPJ-31 Derechos y Deberes de las Víctimas.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 4421 caracteres
[37/234] Procesando: servicio_policia/Formatos Poli

...

   ✓ Guardado: servicio_policia/servicio_policia_extracted/2PR-PR-0005 PRESTAR SERVICIOS EXTRAORDINARIOS DE PROTECCIÓN.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 9865 caracteres
[224/234] Procesando: servicio_policia/Procedimientos Prevención y Control Policial/2PS-GU-0001 GUÍA PARA REPORTE Y RETROALIMENTACIÓN DE INFORMACIÓN EN SALUD.docx
  Procesando con Document Intelligence: servicio_policia/Procedimientos Prevención y Control Policial/2PS-GU-0001 GUÍA PARA REPORTE Y RETROALIMENTACIÓN DE INFORMACIÓN EN SALUD.docx
   ✓ Guardado: servicio_policia/servicio_policia_extracted/2PS-GU-0001 GUÍA PARA REPORTE Y RETROALIMENTACIÓN DE INFORMACIÓN EN SALUD.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 13373 caracteres
[225/234] Procesando: servicio_policia/Procedimientos Prevención y Control Policial/2PS-GU-0002 GUÍA PARA LA PLANIFICACIÓN DEL SERVICIO DE SALUD.docx
  Procesando con Document Intelligence: servicio_policia/Procedimientos Prevención y Control Po

C:\Users\NicolayCastellanosPe\AppData\Local\Temp\ipykernel_41980\3074985718.py:5: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  excel_file = pd.ExcelFile(file_content, engine='openpyxl')


   ✓ Guardado: servicio_policia/servicio_policia_extracted/3PS-PR-0001 ACTIVIDADES DE PREVENCIÓN DEL SECUESTRO Y LA EXTORSIÓN.json
   Páginas/hojas procesadas: 1
   Longitud del contenido: 13382 caracteres
[229/234] Procesando: servicio_policia/RESOLUCION 04180 DEL 09122022_0697 Manual de atención y servicio al ciudadano.pdf
  Procesando con Document Intelligence: servicio_policia/RESOLUCION 04180 DEL 09122022_0697 Manual de atención y servicio al ciudadano.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/RESOLUCION 04180 DEL 09122022_0697 Manual de atención y servicio al ciudadano.json
   Páginas/hojas procesadas: 22
   Longitud del contenido: 69109 caracteres
[230/234] Procesando: servicio_policia/Reglamentación de las medidas posteriores a la aprehensión preventiva, restitución o decomiso flora y fauna.pdf
  Procesando con Document Intelligence: servicio_policia/Reglamentación de las medidas posteriores a la aprehensión preventiva, restitución o decomiso flora y fauna.

C:\Users\NicolayCastellanosPe\AppData\Local\Temp\ipykernel_41980\3074985718.py:5: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  excel_file = pd.ExcelFile(file_content, engine='openpyxl')


   ✓ Guardado: servicio_policia/servicio_policia_extracted/Rubrica_P2P_PONAL.json
   Páginas/hojas procesadas: 4
   Longitud del contenido: 145814 caracteres
[233/234] Procesando: servicio_policia/Seguridad y convivencia ciudadana en Colombia.pdf
  Procesando con Document Intelligence: servicio_policia/Seguridad y convivencia ciudadana en Colombia.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/Seguridad y convivencia ciudadana en Colombia.json
   Páginas/hojas procesadas: 196
   Longitud del contenido: 482970 caracteres
[234/234] Procesando: servicio_policia/ley_1346_de_2009.pdf
  Procesando con Document Intelligence: servicio_policia/ley_1346_de_2009.pdf
   ✓ Guardado: servicio_policia/servicio_policia_extracted/ley_1346_de_2009.json
   Páginas/hojas procesadas: 19
   Longitud del contenido: 79419 caracteres

RESUMEN DE PROCESAMIENTO
Documentos procesados exitosamente: 233
Documentos con errores: 1

Último documento procesado exitosamente:
  Archivo: servicio_policia/l

# Conversión

In [21]:
%pip install pywin32

  Using cached pywin32-311-cp311-cp311-win_amd64.whl.metadata (10 kB)
Using cached pywin32-311-cp311-cp311-win_amd64.whl (9.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import win32com.client as win32
from pathlib import Path

# 📂 Carpeta de entrada (documentos .doc)
INPUT_PATH = Path(r"C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos")

# 📂 Carpeta de salida (se crea automáticamente)
OUTPUT_PATH = Path(r"C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\documents_docx")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

word = win32.Dispatch("Word.Application")
word.Visible = False

for doc_file in INPUT_PATH.rglob("*.doc"):
    # Ignorar archivos temporales de Word
    if doc_file.name.startswith("~$"):
        continue

    # Mantener estructura de carpetas
    relative_path = doc_file.relative_to(INPUT_PATH)
    output_dir = OUTPUT_PATH / relative_path.parent
    output_dir.mkdir(parents=True, exist_ok=True)

    docx_path = output_dir / doc_file.with_suffix(".docx").name

    print(f"Convirtiendo: {doc_file} → {docx_path}")

    try:
        doc = word.Documents.Open(str(doc_file))
        doc.SaveAs(str(docx_path), FileFormat=16)  # 16 = wdFormatXMLDocument
        doc.Close()
    except Exception as e:
        print(f"❌ Error con {doc_file.name}: {e}")

word.Quit()

print("\n✅ Conversión finalizada")
print(f"📁 Documentos convertidos en: {OUTPUT_PATH}")


Convirtiendo: C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\1CS-MA-0001 MANUAL PARA EL SERVICIO EN MANIFESTACIONES Y CONTROL DE DISTURBIOS.doc → C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\documents_docx\1CS-MA-0001 MANUAL PARA EL SERVICIO EN MANIFESTACIONES Y CONTROL DE DISTURBIOS.docx
Convirtiendo: C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\2EI-MA-0002 MANUAL PARA LA ERRADICACIÓN DE CULTIVOS ILÍCITOS.doc → C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\documents_docx\2EI-MA-0002 MANUAL PARA LA ERRADICACIÓN DE CULTIVOS ILÍCITOS.docx
Convirtiendo: C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas\Documentos\3EC-GU-0001 GUÍA PRÁCTICA DEL SISTEMA TÁCTICO BÁSICO POLICIAL.doc → C:\Users\NicolayCastellanosPe\Documents\Policía Nacional\Asistente de IA Normativas